In [ ]:
import pandas as pd
import importlib
import matplotlib as mpl
import matplotlib.pyplot as pp
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
import numpy as np
import scipy as sp
from ipywidgets import widgets
import os
from typing import Optional, List, Tuple
from functools import reduce
import datetime
import _daily_reporters as dr
import functions_for_analyze as ffa

from __visuals__ import _printing_and_display_functions as pad
dailies = os.path.realpath('../../dailies/')
%matplotlib widget

pd.set_option('display.max_colwidth', None)

# Comparing different methods for a single atom - Ar

## Data loading

Here I am loading the data for XG and default, and filtering out a few things

In [ ]:
importlib.reload(ffa)
importlib.reload(pad)
system = 'ar_fully_correlated'

data_f12 = ffa.get_data(
    f'{system}/default/data.csv', ignore_file_cols=False)

data_xg = ffa.get_data(
    f'{system}/xg/data.csv', ignore_file_cols=False)
xg_cols = [col for col in data_xg.columns if 'memory' not in col]
data_xg = data_xg[xg_cols]

pad.header(1, "Data Columns")
pad.header(2, "Default F12 data columns")
data_f12.info()
pad.header(2, "XG data columns")
data_xg.info()
pad.header(2, "XG gamma sets")
display(set(data_xg.gamma_set))

In [ ]:
def filter_data(data, filters):
    for key, values in filters.items():
        data = data[data[key].isin(values)]
    return data

def assign_gamma(xg_data, name='GEM_BETA', which=0):
    get_gamma = lambda row: float(row['gamma_set'].split('_')[0])
    xg_data[name] = xg_data.apply(get_gamma, axis=1)
    return xg_data

def rename_energy(xg_data):
    xg_data.rename(
        columns = {
            'total energy' : 'TOT_ENER',
            'correlation energy' : 'CORR_ENER',
        }, inplace=True
    )
   
    

# Comparing values That should be the same

In [ ]:
f12_filt = dict(
    ansatzes = ['3C(FIX,HY1)'],
    GEM_BETA = [3.0, 1.2],
    core = ["core,0"],
    bases = ['aug-cc-pVTZ']
)

xg_filt_mc1 = dict(
    gamma_set = ['3.00_3.00_3.00', '1.20_1.20_1.20'],
    core = ["core,0"],
    bases = ['aug-cc-pVTZ'],
    xg_maf_folders = ['MAF_CORE_1']
)

xg_filt_mc5 = dict(
    gamma_set = ['3.00_3.00_3.00', '1.20_1.20_1.20'],
    core = ["core,0"],
    bases = ['aug-cc-pVTZ'],
    xg_maf_folders = ['MAF_CORE_5']
)

cols_to_show = ['GEM_BETA', 'TOT_ENER', 'CORR_ENER',
                'gamma_set', 'total energy', 'correlation energy', 'outfile']

f12_f = filter_data(data_f12, f12_filt).loc[:, data_f12.columns.isin(cols_to_show)]
xg1_f = filter_data(data_xg, xg_filt_mc1).loc[:, data_xg.columns.isin(cols_to_show)]
xg5_f = filter_data(data_xg, xg_filt_mc5).loc[:, data_xg.columns.isin(cols_to_show)]
assign_gamma(xg1_f)
assign_gamma(xg5_f)
rename_energy(xg1_f)
rename_energy(xg5_f)

display(f12_f)
display(xg1_f)
display(xg5_f)



In [ ]:

xg_merge = pd.merge(xg1_f, xg5_f, how='inner', on=['GEM_BETA'], suffixes = ['_1', '_5'])
compare_df = pd.merge(f12_f, xg_merge, how='inner', on='GEM_BETA')
cols = [col for col in compare_df if 'file' not in col]
compare_df[cols]

In [ ]:
core = 'core,1'
basis = 'aug-cc-pVTZ'
f12_filt = dict(
    ansatzes = ['3C(FIX,HY1)'],
    GEM_BETA = [3.0, 1.2],
    core = [core],
    bases = [basis]
)

xg_filt_mc1 = dict(
    gamma_set = ['3.00_3.00_3.00', '1.20_1.20_1.20'],
    core = [core],
    bases = [basis],
    xg_maf_folders = ['MAF_CORE_1']
)

xg_filt_mc5 = dict(
    gamma_set = ['3.00_3.00_3.00', '1.20_1.20_1.20'],
    core = [core],
    bases = [basis],
    xg_maf_folders = ['MAF_CORE_5']
)

cols_to_show = ['GEM_BETA', 'TOT_ENER', 'CORR_ENER',
                'gamma_set', 'total energy', 'correlation energy', 'outfile']

f12_f = filter_data(data_f12, f12_filt).loc[:, data_f12.columns.isin(cols_to_show)]
xg1_f = filter_data(data_xg, xg_filt_mc1).loc[:, data_xg.columns.isin(cols_to_show)]
xg5_f = filter_data(data_xg, xg_filt_mc5).loc[:, data_xg.columns.isin(cols_to_show)]
assign_gamma(xg1_f)
assign_gamma(xg5_f)
rename_energy(xg1_f)
rename_energy(xg5_f)

display(f12_f)
display(xg1_f)
display(xg5_f)



In [ ]:

xg_merge = pd.merge(xg1_f, xg5_f, how='inner', on=['GEM_BETA'], suffixes = ['_1', '_5'])
compare_df = pd.merge(f12_f, xg_merge, how='inner', on='GEM_BETA')
cols = [col for col in compare_df if 'file' not in col]
compare_df[cols]